In [1]:
import os
import wandb

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import numpy as np
import pandas as pd
import random
from tqdm import *

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets
import torchvision.transforms as transforms

try:
    from torchinfo import summary
except:
    print("[INFO] Couldn't find torchinfo... installing it.")
    !pip install -q torchinfo
    from torchinfo import summary

from PIL import Image

import matplotlib.pyplot as plt
%matplotlib inline

[INFO] Couldn't find torchinfo... installing it.


### Зафиксируем seed для воспроизводимости

In [2]:
def seed_everything(seed):
    random.seed(seed) # фиксируем генератор случайных чисел
    os.environ['PYTHONHASHSEED'] = str(seed) # фиксируем заполнения хешей
    np.random.seed(seed) # фиксируем генератор случайных чисел numpy
    torch.manual_seed(seed) # фиксируем генератор случайных чисел pytorch
    torch.cuda.manual_seed(seed) # фиксируем генератор случайных чисел для GPU
    torch.backends.cudnn.deterministic = True # выбираем только детерминированные алгоритмы (для сверток)
    torch.backends.cudnn.benchmark = False # фиксируем алгоритм вычисления сверток

### Задаем параметры (конфиг)

In [3]:
class CFG:
  api = ""
  project = "KOLESA"
  entity = ""
  num_epochs = 10
  train_batch_size = 32
  test_batch_size = 512
  num_workers = 0
  lr = 3e-4
  seed = 42
  classes = ('sedan', 'SUV')
  wandb = False

In [4]:
seed_everything(CFG.seed)

In [5]:
# Переведем наш класс с параметрами в словарь

def class2dict(f):
  return dict((name, getattr(f, name)) for name in dir(f) if not name.startswith('__'))

### Импортируем наш датасет с картинками

In [6]:
try:
  from google.colab import drive
  drive.mount('/content/drive')
  !cp -r /content/drive/MyDrive/gp5_images/data /content/data

  base_dir = '/content/data'
except:
  base_dir = '../data'

from pathlib import Path

base_dir = Path(base_dir)

Mounted at /content/drive


In [7]:
df = pd.read_csv(base_dir / 'prepared_df_images.csv')
df

,kolesa_id,body_type,img_filename,target
0,220599328,Седан,220599328.webp,0
1,222486528,Седан,222486528.webp,0
2,218695550,Внедорожник,218695550.webp,1
3,214781208,Седан,214781208.webp,0
4,222346737,Седан,222346737.webp,0
...,...,...,...,...
4428,222478046,Седан,222478046.webp,0
4429,222704181,Седан,222704181.webp,0
4430,220000872,Седан,220000872.webp,0
4431,222542135,Седан,222542135.webp,0


In [9]:
train_df, test_all_df = train_test_split(df, test_size=0.3, stratify=df['target'], random_state=CFG.seed)
val_df, test_df = train_test_split(test_all_df, test_size=0.5, stratify=test_all_df['target'], random_state=CFG.seed)

In [10]:
# https://stackoverflow.com/questions/65231299/load-csv-and-image-dataset-in-pytorch

class KolesaDataset(torch.utils.data.Dataset):
    def __init__(self, df, images_folder = base_dir / 'images', transform = None):
        # т.к. после train_test_split индексы сохряняются из исходного датасета, то мы их сбрасываем и удаляем колонку со старыми индексами для корректной работы __getitem___
        self.df = df.reset_index().drop(columns=['index'])
        self.images_folder = images_folder
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        filename = self.df.loc[index, 'img_filename']
        label = self.df.loc[index, 'target']
        image = Image.open(os.path.join(self.images_folder, filename)).convert('RGB')

        if self.transform is not None:
            image = self.transform(image)

        return image, label



### Функции обучения и инференса

In [11]:
# функция обучения
def train(model, device, train_loader, optimizer, criterion, epoch, WANDB):
    model.train()
    train_loss = 0
    correct = 0

    n_ex = len(train_loader)

    for batch_idx, (data, target) in tqdm(enumerate(train_loader), total=n_ex):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad() # обнуляем градиенты
        output = model(data)
        pred = output.argmax(dim=1, keepdim=True)
        correct += pred.eq(target.view_as(pred)).sum().item()
        train_loss = criterion(output, target) # считаем лосс
        train_loss.backward() # обратный проход
        optimizer.step() # делаем шаг оптимизатором

    tqdm.write('\nTrain set: Average loss: {:.4f}, Accuracy: {:.2f}%'.format(
        train_loss, 100. * correct / len(train_loader.dataset)))

    # логируем функцию потерь и точность
    if WANDB:
        wandb.log({'train_loss': train_loss,
                   'train_accuracy': correct / len(train_loader.dataset)})

In [12]:
# функция инференса
def test(model, device, test_loader, criterion, WANDB):
    model.eval()
    test_loss = 0
    correct = 0

    preds = []
    targets = []
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss = criterion(output, target)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

            # https://stackoverflow.com/questions/53467215/convert-pytorch-cuda-tensor-to-numpy-array
            preds.extend(pred.cpu().numpy())
            targets.extend(target.cpu().numpy())

    # https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html
    report = classification_report(targets, preds, target_names=CFG.classes)

    tqdm.write('Test set: Average loss: {:.4f}, Accuracy: {:.2f}%'.format(
        test_loss, 100. * correct / len(test_loader.dataset)))

    tqdm.write('\nClassification report:\n' + report)

    if WANDB:
        wandb.log({'test_loss': test_loss,
                   'test_accuracy': correct / len(test_loader.dataset)})

# функция инференса
def val(model, device, val_loader, criterion, WANDB):
    model.eval()
    val_loss = 0
    correct = 0

    preds = []
    targets = []
    with torch.no_grad():
        for data, target in val_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            val_loss = criterion(output, target)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

            preds.extend(pred.cpu().numpy())
            targets.extend(target.cpu().numpy())

    report = classification_report(targets, preds, target_names=CFG.classes)

    tqdm.write('Val set: Average loss: {:.4f}, Accuracy: {:.2f}%'.format(
        val_loss, 100. * correct / len(val_loader.dataset)))

    tqdm.write('\nClassification report:\n' + report)

    if WANDB:
        wandb.log({'val_loss': val_loss,
                   'val_accuracy': correct / len(val_loader.dataset)})

### Основная функция для прогона моделей

In [13]:
def main_kolesa(model):

    if CFG.wandb:
        os.environ["WANDB_API_KEY"] = CFG.api
        wandb.init(project=CFG.project, entity=CFG.entity, reinit=True, config=class2dict(CFG))

    seed_everything(CFG.seed)
    # https://stackoverflow.com/questions/63423463/using-pytorch-cuda-on-macbook-pro
    # т.к. на macbook на их процессорах apple silicon нет cuda (только для карт nvidia), то используем альтеранативу, но если есть cuda - то используем ее
    if torch.cuda.is_available():
        device = torch.device('cuda')
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
    else:
        device = torch.device('cpu')

    kwargs = {'num_workers': CFG.num_workers, 'pin_memory': torch.cuda.is_available()}

    # наши картинки в исходном размере 750*470 + если домножить на число каналов (3), то получится около миллиарда параметра на вход.
    # Это довольно много так что пропорционально уменьшим размерность. У нас 750 к 470 это, можно сказать, 1.6 к 1. Уменьшим 750 до 160 => 470 -> 96.
    # будем использовать различные методы аугментации https://docs.pytorch.org/vision/0.13/transforms.html
    train_transform = transforms.Compose([
        transforms.Resize((96, 160)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225)
        )
        ])

    test_transform = transforms.Compose([
        transforms.Resize((96, 160)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225)
        )
        ])

    train_ds = KolesaDataset(train_df, transform=train_transform)
    val_ds = KolesaDataset(val_df, transform=test_transform)
    test_ds = KolesaDataset(test_df, transform=test_transform)

    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=CFG.train_batch_size, **kwargs)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=CFG.test_batch_size, **kwargs)
    test_loader = torch.utils.data.DataLoader(test_ds, batch_size=CFG.test_batch_size, **kwargs)

    model = model.to(device)

    if CFG.wandb:
        wandb.watch(model, log='all')

    optimizer = optim.Adam(model.parameters(),
                          lr=CFG.lr)

    criterion = nn.CrossEntropyLoss()

    for epoch in range(1, CFG.num_epochs + 1):
        print('\nEpoch:', epoch)
        train(model, device, train_loader, optimizer, criterion, epoch, CFG.wandb)
        val(model, device, val_loader, criterion, CFG.wandb)

    print('Training is ended!')

## Fully Connected в качестве baseline

Возьмем для примера определение весов нормальным распределением

In [ ]:
# определяем полносвязанную сеть
class FC_Net(nn.Module):
    def __init__(self, constant_weight=None, normal=False,
                 xavier_uniform=False, he_normal=False):
        super(FC_Net, self).__init__()

        self.fc1 = nn.Linear(160*96*3, 1024)
        self.fc2 = nn.Linear(1024, 512)
        self.fc3 = nn.Linear(512, 128)
        self.fc4 = nn.Linear(128, 2)

        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = x.view(-1, 160*96*3)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.fc4(x)
        return x

In [ ]:
model_baseline = FC_Net()

In [ ]:
summary(model=model_baseline,
        input_size=(32, 3, 96, 160), # входной батч
        col_names=["input_size", "output_size", "num_params", "trainable"], # что хотим посмотреть
        col_width=20
)

Layer (type:depth-idx)                   Input Shape          Output Shape         Param #              Trainable
FC_Net                                   [32, 3, 96, 160]     [32, 2]              --                   True
├─Linear: 1-1                            [32, 46080]          [32, 1024]           47,186,944           True
├─Linear: 1-2                            [32, 1024]           [32, 512]            524,800              True
├─Linear: 1-3                            [32, 512]            [32, 128]            65,664               True
├─Linear: 1-4                            [32, 128]            [32, 2]              258                  True
Total params: 47,777,666
Trainable params: 47,777,666
Non-trainable params: 0
Total mult-adds (Units.GIGABYTES): 1.53
Input size (MB): 5.90
Forward/backward pass size (MB): 0.43
Params size (MB): 191.11
Estimated Total Size (MB): 197.44

In [ ]:
main_kolesa(model_baseline)


Epoch: 1


100%|██████████| 97/97 [00:22<00:00,  4.28it/s]



Train set: Average loss: 200857.7188, Accuracy: 54.04%
Val set: Average loss: 133297.0781, Accuracy: 54.74%

Classification report:
              precision    recall  f1-score   support

       sedan       0.66      0.58      0.62       418
         SUV       0.41      0.50      0.45       247

    accuracy                           0.55       665
   macro avg       0.54      0.54      0.53       665
weighted avg       0.57      0.55      0.55       665


Epoch: 2


100%|██████████| 97/97 [00:16<00:00,  6.00it/s]



Train set: Average loss: 100811.0938, Accuracy: 56.91%
Val set: Average loss: 111838.9922, Accuracy: 56.24%

Classification report:
              precision    recall  f1-score   support

       sedan       0.66      0.63      0.64       418
         SUV       0.42      0.45      0.43       247

    accuracy                           0.56       665
   macro avg       0.54      0.54      0.54       665
weighted avg       0.57      0.56      0.57       665


Epoch: 3


...


Train set: Average loss: 64285.5820, Accuracy: 62.68%
Val set: Average loss: 83032.4453, Accuracy: 55.19%

Classification report:
              precision    recall  f1-score   support

       sedan       0.65      0.61      0.63       418
         SUV       0.41      0.45      0.42       247

    accuracy                           0.55       665
   macro avg       0.53      0.53      0.53       665
weighted avg       0.56      0.55      0.56       665


Epoch: 9


100%|██████████| 97/97 [00:15<00:00,  6.08it/s]



Train set: Average loss: 49918.4648, Accuracy: 64.87%
Val set: Average loss: 75215.9766, Accuracy: 56.24%

Classification report:
              precision    recall  f1-score   support

       sedan       0.68      0.57      0.62       418
         SUV       0.43      0.55      0.48       247

    accuracy                           0.56       665
   macro avg       0.56      0.56      0.55       665
weighted avg       0.59      0.56      0.57       665


Epoch: 10


100%|██████████| 97/97 [00:15<00:00,  6.16it/s]



Train set: Average loss: 31791.6484, Accuracy: 63.87%
Val set: Average loss: 77674.0156, Accuracy: 54.74%

Classification report:
              precision    recall  f1-score   support

       sedan       0.66      0.57      0.61       418
         SUV       0.41      0.51      0.46       247

    accuracy                           0.55       665
   macro avg       0.54      0.54      0.53       665
weighted avg       0.57      0.55      0.55       665

Final on test
Test set: Average loss: 64038.9023, Accuracy: 54.74%

Classification report:
              precision    recall  f1-score   support

       sedan       0.67      0.56      0.61       419
         SUV       0.41      0.52      0.46       246

    accuracy                           0.55       665
   macro avg       0.54      0.54      0.54       665
weighted avg       0.57      0.55      0.55       665

Training is ended!


Модель сильно переобучилась. Посмотрим далее сверточную сеть

## Простой CNN

In [ ]:
class CNN_Net(nn.Module):
    def __init__(self):
        super(CNN_Net, self).__init__()

        self.conv1 = torch.nn.Conv2d(3, 16, 3, padding=1)
        self.act1  = torch.nn.ReLU()
        self.pool1 = torch.nn.MaxPool2d(2, 2) # 48 80

        self.conv2 = torch.nn.Conv2d(16, 32, 3, padding=1)
        self.act2  = torch.nn.ReLU()
        self.pool2 = torch.nn.MaxPool2d(2, 2) # 24 40

        self.conv3 = torch.nn.Conv2d(32, 64, 3, padding=1)
        self.act3  = torch.nn.ReLU()
        self.pool3 = torch.nn.MaxPool2d(2, 2) # 12 20

        self.fc1   = torch.nn.Linear(12 * 20 * 64, 256)
        self.act4  = torch.nn.Tanh()

        self.fc2   = torch.nn.Linear(256, 64)
        self.act5  = torch.nn.Tanh()

        self.fc3   = torch.nn.Linear(64, 2)

    def forward(self, x):
        x = self.conv1(x)
        x = self.act1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.act2(x)
        x = self.pool2(x)

        x = self.conv3(x)
        x = self.act3(x)
        x = self.pool3(x)

        x = x.view(x.size(0), x.size(1) * x.size(2) * x.size(3))
        x = self.fc1(x)
        x = self.act4(x)
        x = self.fc2(x)
        x = self.act5(x)
        x = self.fc3(x)

        return x

In [ ]:
model_cnn_1 = CNN_Net()

In [ ]:
summary(model=model_cnn_1,
        input_size=(32, 3, 96, 160), # входной батч
        col_names=["input_size", "output_size", "num_params", "trainable"], # что хотим посмотреть
        col_width=20
)

Layer (type:depth-idx)                   Input Shape          Output Shape         Param #              Trainable
CNN_Net                                  [32, 3, 96, 160]     [32, 2]              --                   True
├─Conv2d: 1-1                            [32, 3, 96, 160]     [32, 16, 96, 160]    448                  True
├─ReLU: 1-2                              [32, 16, 96, 160]    [32, 16, 96, 160]    --                   --
├─MaxPool2d: 1-3                         [32, 16, 96, 160]    [32, 16, 48, 80]     --                   --
├─Conv2d: 1-4                            [32, 16, 48, 80]     [32, 32, 48, 80]     4,640                True
├─ReLU: 1-5                              [32, 32, 48, 80]     [32, 32, 48, 80]     --                   --
├─MaxPool2d: 1-6                         [32, 32, 48, 80]     [32, 32, 24, 40]     --                   --
├─Conv2d: 1-7                            [32, 32, 24, 40]     [32, 64, 24, 40]     18,496               True
├─ReLU: 1-8           

In [ ]:
main_kolesa(model_cnn_1)


Epoch: 1


100%|██████████| 97/97 [00:14<00:00,  6.53it/s]



Train set: Average loss: 0.7866, Accuracy: 62.07%


/Users/ksa/CodeProjects/gp_5_DL/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/ksa/CodeProjects/gp_5_DL/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/ksa/CodeProjects/gp_5_DL/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"

Val set: Average loss: 0.6645, Accuracy: 62.86%

Classification report:
              precision    recall  f1-score   support

       sedan       0.63      1.00      0.77       418
         SUV       0.00      0.00      0.00       247

    accuracy                           0.63       665
   macro avg       0.31      0.50      0.39       665
weighted avg       0.40      0.63      0.49       665


Epoch: 2


...


Train set: Average loss: 0.6293, Accuracy: 69.06%
Val set: Average loss: 0.6262, Accuracy: 67.52%

Classification report:
              precision    recall  f1-score   support

       sedan       0.73      0.77      0.75       418
         SUV       0.57      0.51      0.54       247

    accuracy                           0.68       665
   macro avg       0.65      0.64      0.64       665
weighted avg       0.67      0.68      0.67       665


Epoch: 9


100%|██████████| 97/97 [00:14<00:00,  6.58it/s]



Train set: Average loss: 0.6057, Accuracy: 70.67%
Val set: Average loss: 0.6505, Accuracy: 66.17%

Classification report:
              precision    recall  f1-score   support

       sedan       0.74      0.72      0.73       418
         SUV       0.54      0.57      0.55       247

    accuracy                           0.66       665
   macro avg       0.64      0.64      0.64       665
weighted avg       0.66      0.66      0.66       665


Epoch: 10


100%|██████████| 97/97 [00:14<00:00,  6.52it/s]



Train set: Average loss: 0.6707, Accuracy: 70.74%
Val set: Average loss: 0.6077, Accuracy: 67.37%

Classification report:
              precision    recall  f1-score   support

       sedan       0.72      0.78      0.75       418
         SUV       0.57      0.50      0.53       247

    accuracy                           0.67       665
   macro avg       0.65      0.64      0.64       665
weighted avg       0.67      0.67      0.67       665

Final on test
Test set: Average loss: 0.6560, Accuracy: 68.27%

Classification report:
              precision    recall  f1-score   support

       sedan       0.74      0.77      0.75       419
         SUV       0.58      0.54      0.56       246

    accuracy                           0.68       665
   macro avg       0.66      0.65      0.65       665
weighted avg       0.68      0.68      0.68       665

Training is ended!


Не сильно далеко ушли от FC, по крайней мере, пока качество не самое приятное

## CNN с регуляризацией

Теперь попробуем добавить регуляризацию, а именно известные нам BatchNorm и Dropout.

Также поменяем функции активации https://deepmachinelearning.ru/docs/Neural-networks/MLP/Activation-functions. В частности попробуем LeakyReLU.

И наконец добавим побольше сверточных слоев.

In [ ]:
class CNNReg_Net(nn.Module):
    def __init__(self):
        super(CNNReg_Net, self).__init__()

        self.conv3 = torch.nn.Conv2d(3, 32, 3, padding=1)
        self.bn3   = torch.nn.BatchNorm2d(32)
        self.act3  = torch.nn.LeakyReLU()

        self.conv4 = torch.nn.Conv2d(32, 32, 3, padding=1)
        self.bn4   = torch.nn.BatchNorm2d(32)
        self.act4  = torch.nn.LeakyReLU()
        self.pool2 = torch.nn.MaxPool2d(2, 2) # 24 40

        self.conv5 = torch.nn.Conv2d(32, 64, 3, padding=1)
        self.bn5   = torch.nn.BatchNorm2d(64)
        self.act5  = torch.nn.LeakyReLU()

        self.conv6 = torch.nn.Conv2d(64, 64, 3, padding=1)
        self.bn6   = torch.nn.BatchNorm2d(64)
        self.act6  = torch.nn.LeakyReLU()
        self.pool3 = torch.nn.MaxPool2d(2, 2) # 12 20

        self.drop1 = torch.nn.Dropout2d(0.10)

        self.conv7 = torch.nn.Conv2d(64, 128, 3, padding=1)
        self.bn7   = torch.nn.BatchNorm2d(128)
        self.act7  = torch.nn.LeakyReLU()
        self.drop2 = torch.nn.Dropout2d(0.15)

        self.conv8 = torch.nn.Conv2d(128, 128, 3, padding=1)
        self.bn8   = torch.nn.BatchNorm2d(128)
        self.act8  = torch.nn.LeakyReLU()
        self.pool4 = torch.nn.MaxPool2d(2, 2)
        self.drop3 = torch.nn.Dropout2d(0.2)

        self.conv9 = torch.nn.Conv2d(128, 256, 3, padding=1)
        self.bn9   = torch.nn.BatchNorm2d(256)
        self.act9  = torch.nn.LeakyReLU()
        self.pool5 = torch.nn.MaxPool2d(2, 2)
        self.drop4 = torch.nn.Dropout2d(0.25)

        self.fc1   = torch.nn.Linear(6 * 10 * 256, 256)
        self.bn10  = torch.nn.BatchNorm1d(256)
        self.act10 = torch.nn.LeakyReLU()
        self.drop5 = torch.nn.Dropout(0.3)

        self.fc2   = torch.nn.Linear(256, 64)
        self.act11 = torch.nn.LeakyReLU()
        self.drop6 = torch.nn.Dropout(0.2)

        self.fc3   = torch.nn.Linear(64, 2)

    def forward(self, x):
        x = self.conv3(x)
        x = self.bn3(x)
        x = self.act3(x)

        x = self.conv4(x)
        x = self.bn4(x)
        x = self.act4(x)
        x = self.pool2(x)

        x = self.conv5(x)
        x = self.bn5(x)
        x = self.act5(x)

        x = self.conv6(x)
        x = self.bn6(x)
        x = self.act6(x)
        x = self.pool3(x)
        x = self.drop1(x)

        x = self.conv7(x)
        x = self.bn7(x)
        x = self.act7(x)
        x = self.drop2(x)

        x = self.conv8(x)
        x = self.bn8(x)
        x = self.act8(x)
        x = self.pool4(x)
        x = self.drop3(x)

        x = self.conv9(x)
        x = self.bn9(x)
        x = self.act9(x)
        x = self.pool5(x)
        x = self.drop4(x)

        x = x.view(x.size(0), x.size(1) * x.size(2) * x.size(3))
        x = self.fc1(x)
        x = self.bn10(x)
        x = self.act10(x)
        x = self.drop5(x)

        x = self.fc2(x)
        x = self.act11(x)
        x = self.drop6(x)

        x = self.fc3(x)

        return x

In [ ]:
model_cnn_2 = CNNReg_Net()

In [ ]:
summary(model=model_cnn_2,
        input_size=(32, 3, 96, 160), # входной батч
        col_names=["input_size", "output_size", "num_params", "trainable"], # что хотим посмотреть
        col_width=20
)

Layer (type:depth-idx)                   Input Shape          Output Shape         Param #              Trainable
CNNReg_Net                               [32, 3, 96, 160]     [32, 2]              --                   True
├─Conv2d: 1-1                            [32, 3, 96, 160]     [32, 32, 96, 160]    896                  True
├─BatchNorm2d: 1-2                       [32, 32, 96, 160]    [32, 32, 96, 160]    64                   True
├─LeakyReLU: 1-3                         [32, 32, 96, 160]    [32, 32, 96, 160]    --                   --
├─Conv2d: 1-4                            [32, 32, 96, 160]    [32, 32, 96, 160]    9,248                True
├─BatchNorm2d: 1-5                       [32, 32, 96, 160]    [32, 32, 96, 160]    64                   True
├─LeakyReLU: 1-6                         [32, 32, 96, 160]    [32, 32, 96, 160]    --                   --
├─MaxPool2d: 1-7                         [32, 32, 96, 160]    [32, 32, 48, 80]     --                   --
├─Conv2d: 1-8       

In [ ]:
main_kolesa(model_cnn_2)


Epoch: 1


100%|██████████| 97/97 [00:18<00:00,  5.18it/s]



Train set: Average loss: 0.7882, Accuracy: 61.07%
Val set: Average loss: 0.6315, Accuracy: 65.11%

Classification report:
              precision    recall  f1-score   support

       sedan       0.66      0.93      0.77       418
         SUV       0.60      0.18      0.28       247

    accuracy                           0.65       665
   macro avg       0.63      0.56      0.52       665
weighted avg       0.64      0.65      0.59       665


Epoch: 2


100%|██████████| 97/97 [00:20<00:00,  4.79it/s]



Train set: Average loss: 0.7519, Accuracy: 64.00%
Val set: Average loss: 0.6184, Accuracy: 65.26%

Classification report:
              precision    recall  f1-score   support

       sedan       0.70      0.78      0.74       418
         SUV       0.54      0.43      0.48       247

    accuracy                           0.65       665
   macro avg       0.62      0.61      0.61       665
weighted avg       0.64      0.65      0.64       665


Epoch: 3


...


Train set: Average loss: 0.6738, Accuracy: 70.61%
Val set: Average loss: 0.5662, Accuracy: 70.83%

Classification report:
              precision    recall  f1-score   support

       sedan       0.77      0.77      0.77       418
         SUV       0.61      0.61      0.61       247

    accuracy                           0.71       665
   macro avg       0.69      0.69      0.69       665
weighted avg       0.71      0.71      0.71       665


Epoch: 9


100%|██████████| 97/97 [00:19<00:00,  4.99it/s]



Train set: Average loss: 0.7744, Accuracy: 72.28%
Val set: Average loss: 0.6086, Accuracy: 66.62%

Classification report:
              precision    recall  f1-score   support

       sedan       0.80      0.63      0.70       418
         SUV       0.54      0.73      0.62       247

    accuracy                           0.67       665
   macro avg       0.67      0.68      0.66       665
weighted avg       0.70      0.67      0.67       665


Epoch: 10


100%|██████████| 97/97 [00:18<00:00,  5.35it/s]



Train set: Average loss: 0.7415, Accuracy: 72.54%
Val set: Average loss: 0.5212, Accuracy: 73.53%

Classification report:
              precision    recall  f1-score   support

       sedan       0.79      0.79      0.79       418
         SUV       0.64      0.64      0.64       247

    accuracy                           0.74       665
   macro avg       0.72      0.72      0.72       665
weighted avg       0.74      0.74      0.74       665

Final on test
Test set: Average loss: 0.5487, Accuracy: 72.48%

Classification report:
              precision    recall  f1-score   support

       sedan       0.79      0.76      0.78       419
         SUV       0.62      0.66      0.64       246

    accuracy                           0.72       665
   macro avg       0.71      0.71      0.71       665
weighted avg       0.73      0.72      0.73       665

Training is ended!


In [ ]:
main_kolesa(model_cnn_2)


Epoch: 1


100%|██████████| 97/97 [00:18<00:00,  5.25it/s]



Train set: Average loss: 0.6372, Accuracy: 74.28%
Val set: Average loss: 0.5271, Accuracy: 72.18%

Classification report:
              precision    recall  f1-score   support

       sedan       0.78      0.77      0.78       418
         SUV       0.62      0.64      0.63       247

    accuracy                           0.72       665
   macro avg       0.70      0.71      0.70       665
weighted avg       0.72      0.72      0.72       665


Epoch: 2


100%|██████████| 97/97 [00:18<00:00,  5.36it/s]



Train set: Average loss: 0.6632, Accuracy: 75.67%
Val set: Average loss: 0.5053, Accuracy: 71.88%

Classification report:
              precision    recall  f1-score   support

       sedan       0.80      0.74      0.77       418
         SUV       0.61      0.68      0.64       247

    accuracy                           0.72       665
   macro avg       0.70      0.71      0.71       665
weighted avg       0.73      0.72      0.72       665


Epoch: 3


...


Train set: Average loss: 0.5836, Accuracy: 81.70%
Val set: Average loss: 0.6016, Accuracy: 66.77%

Classification report:
              precision    recall  f1-score   support

       sedan       0.85      0.57      0.68       418
         SUV       0.53      0.83      0.65       247

    accuracy                           0.67       665
   macro avg       0.69      0.70      0.67       665
weighted avg       0.73      0.67      0.67       665


Epoch: 9


100%|██████████| 97/97 [00:19<00:00,  4.96it/s]



Train set: Average loss: 0.5354, Accuracy: 83.24%
Val set: Average loss: 0.8675, Accuracy: 58.95%

Classification report:
              precision    recall  f1-score   support

       sedan       0.85      0.42      0.56       418
         SUV       0.47      0.87      0.61       247

    accuracy                           0.59       665
   macro avg       0.66      0.65      0.59       665
weighted avg       0.71      0.59      0.58       665


Epoch: 10


100%|██████████| 97/97 [00:19<00:00,  4.95it/s]



Train set: Average loss: 0.6861, Accuracy: 82.92%
Val set: Average loss: 0.6226, Accuracy: 69.02%

Classification report:
              precision    recall  f1-score   support

       sedan       0.85      0.61      0.71       418
         SUV       0.56      0.82      0.66       247

    accuracy                           0.69       665
   macro avg       0.70      0.72      0.69       665
weighted avg       0.74      0.69      0.69       665

Final on test
Test set: Average loss: 0.6902, Accuracy: 65.56%

Classification report:
              precision    recall  f1-score   support

       sedan       0.85      0.55      0.67       419
         SUV       0.52      0.84      0.64       246

    accuracy                           0.66       665
   macro avg       0.69      0.69      0.66       665
weighted avg       0.73      0.66      0.66       665

Training is ended!


In [ ]:
main_kolesa(model_cnn_2)


Epoch: 1


100%|██████████| 97/97 [00:21<00:00,  4.41it/s]



Train set: Average loss: 0.4725, Accuracy: 86.46%
Val set: Average loss: 0.5756, Accuracy: 71.88%

Classification report:
              precision    recall  f1-score   support

       sedan       0.83      0.69      0.76       418
         SUV       0.59      0.76      0.67       247

    accuracy                           0.72       665
   macro avg       0.71      0.73      0.71       665
weighted avg       0.74      0.72      0.72       665


Epoch: 2


100%|██████████| 97/97 [00:20<00:00,  4.83it/s]



Train set: Average loss: 0.5105, Accuracy: 88.08%
Val set: Average loss: 0.6803, Accuracy: 67.07%

Classification report:
              precision    recall  f1-score   support

       sedan       0.86      0.56      0.68       418
         SUV       0.54      0.85      0.66       247

    accuracy                           0.67       665
   macro avg       0.70      0.71      0.67       665
weighted avg       0.74      0.67      0.67       665


Epoch: 3


...


Train set: Average loss: 0.3488, Accuracy: 91.81%
Val set: Average loss: 0.7280, Accuracy: 68.27%

Classification report:
              precision    recall  f1-score   support

       sedan       0.83      0.63      0.71       418
         SUV       0.55      0.78      0.65       247

    accuracy                           0.68       665
   macro avg       0.69      0.70      0.68       665
weighted avg       0.72      0.68      0.69       665


Epoch: 9


100%|██████████| 97/97 [00:20<00:00,  4.69it/s]



Train set: Average loss: 0.3443, Accuracy: 93.07%
Val set: Average loss: 0.8842, Accuracy: 68.12%

Classification report:
              precision    recall  f1-score   support

       sedan       0.83      0.62      0.71       418
         SUV       0.55      0.78      0.64       247

    accuracy                           0.68       665
   macro avg       0.69      0.70      0.68       665
weighted avg       0.72      0.68      0.69       665


Epoch: 10


100%|██████████| 97/97 [00:22<00:00,  4.27it/s]



Train set: Average loss: 0.7052, Accuracy: 91.72%
Val set: Average loss: 0.7649, Accuracy: 67.22%

Classification report:
              precision    recall  f1-score   support

       sedan       0.85      0.58      0.69       418
         SUV       0.54      0.83      0.65       247

    accuracy                           0.67       665
   macro avg       0.70      0.71      0.67       665
weighted avg       0.74      0.67      0.68       665

Final on test
Test set: Average loss: 0.9292, Accuracy: 64.96%

Classification report:
              precision    recall  f1-score   support

       sedan       0.87      0.52      0.65       419
         SUV       0.52      0.87      0.65       246

    accuracy                           0.65       665
   macro avg       0.69      0.69      0.65       665
weighted avg       0.74      0.65      0.65       665

Training is ended!


Вышло уже в разы лучше

## FineTuned AlexNet

Далее попробуем в качестве другого подхода взять уже ранее предобученную модель. Возьмем AlexNet

Чтобы у каждой модели сохранялся артефакт, добавим маленькую вспомогательную функцию. `main_kolesa` логирует loss/accuracy по эпохам в W&B сам, а эта функция дописывает в тот же запуск файл с весами.

In [14]:
def save_model_artifact(model, name):
    os.makedirs('models', exist_ok=True) # Создаем папку models в рабочей директории, куда будем складывать файлы с весами. exist_ok=True позволяет не падать из-а ошибки, если папка уже существвет
    path = f'models/{name}.pt' #путь к файлу в зависимости от имени модели (передаем в начале), .pt -расширение для объектов pytorch
    torch.save(model.state_dict(), path) #сохраняем параметры модели
    if CFG.wandb:
        artifact = wandb.Artifact(name, type='model') #создаем артефакт-контейннер
        artifact.add_file(path) #добавляем в него файл с параметрами модели (брали выше)
        wandb.log_artifact(artifact) #загружаем артефакт и привящываем к текущему щапуску
        wandb.finish()  # закрываем текущий запуск
    print(f'[OK] модель сохранена: {path}')

In [15]:
from torchvision.models import alexnet
from torchvision.models import AlexNet_Weights

num_in_features = 9216  # Alex net после сверток выдает 256 каналов, а пуллинг сжимает картинку до 6 x 6
num_out_features = 2    # Sedan и SUV

CFG.lr = 1e-4 #learning rate понижаем для более корректного шага модели

model_alexnet = alexnet(weights=AlexNet_Weights.DEFAULT) #DEFAULT - рекомендованный вариант
model_alexnet.classifier = torch.nn.Linear(num_in_features, num_out_features) #alexnet.classifier по умолчанию выдает 1000 классов, а нам нужно 2, поэтому сокращаем

Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 184MB/s]


In [16]:
summary(model=model_alexnet,
        input_size=(32, 3, 96, 160), #картинкики, канал, высота и ширина
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20
) #прогоняем фиктивный батч через модель

Layer (type:depth-idx)                   Input Shape          Output Shape         Param #              Trainable
AlexNet                                  [32, 3, 96, 160]     [32, 2]              --                   True
├─Sequential: 1-1                        [32, 3, 96, 160]     [32, 256, 2, 4]      --                   True
│    └─Conv2d: 2-1                       [32, 3, 96, 160]     [32, 64, 23, 39]     23,296               True
│    └─ReLU: 2-2                         [32, 64, 23, 39]     [32, 64, 23, 39]     --                   --
│    └─MaxPool2d: 2-3                    [32, 64, 23, 39]     [32, 64, 11, 19]     --                   --
│    └─Conv2d: 2-4                       [32, 64, 11, 19]     [32, 192, 11, 19]    307,392              True
│    └─ReLU: 2-5                         [32, 192, 11, 19]    [32, 192, 11, 19]    --                   --
│    └─MaxPool2d: 2-6                    [32, 192, 11, 19]    [32, 192, 5, 9]      --                   --
│    └─Conv2d: 2-7    

In [17]:
main_kolesa(model_alexnet)
save_model_artifact(model_alexnet, 'AlexNet_finetune')


Epoch: 1


100%|██████████| 97/97 [00:36<00:00,  2.63it/s]



Train set: Average loss: 0.6713, Accuracy: 64.78%
Val set: Average loss: 0.5624, Accuracy: 69.92%

Classification report:
              precision    recall  f1-score   support

       sedan       0.79      0.72      0.75       418
         SUV       0.58      0.67      0.62       247

    accuracy                           0.70       665
   macro avg       0.68      0.69      0.69       665
weighted avg       0.71      0.70      0.70       665


Epoch: 2


100%|██████████| 97/97 [00:37<00:00,  2.60it/s]



Train set: Average loss: 0.5156, Accuracy: 72.45%
Val set: Average loss: 0.5699, Accuracy: 69.77%

Classification report:
              precision    recall  f1-score   support

       sedan       0.84      0.64      0.73       418
         SUV       0.57      0.80      0.66       247

    accuracy                           0.70       665
   macro avg       0.71      0.72      0.69       665
weighted avg       0.74      0.70      0.70       665


Epoch: 3


...


Train set: Average loss: 0.3382, Accuracy: 89.20%
Val set: Average loss: 0.4385, Accuracy: 79.40%

Classification report:
              precision    recall  f1-score   support

       sedan       0.83      0.85      0.84       418
         SUV       0.74      0.70      0.72       247

    accuracy                           0.79       665
   macro avg       0.78      0.77      0.78       665
weighted avg       0.79      0.79      0.79       665


Epoch: 9


100%|██████████| 97/97 [00:37<00:00,  2.59it/s]



Train set: Average loss: 0.1706, Accuracy: 88.56%
Val set: Average loss: 0.4119, Accuracy: 78.35%

Classification report:
              precision    recall  f1-score   support

       sedan       0.85      0.79      0.82       418
         SUV       0.69      0.77      0.72       247

    accuracy                           0.78       665
   macro avg       0.77      0.78      0.77       665
weighted avg       0.79      0.78      0.79       665


Epoch: 10


100%|██████████| 97/97 [00:36<00:00,  2.65it/s]



Train set: Average loss: 0.1649, Accuracy: 89.72%
Val set: Average loss: 0.3900, Accuracy: 78.50%

Classification report:
              precision    recall  f1-score   support

       sedan       0.84      0.81      0.82       418
         SUV       0.70      0.75      0.72       247

    accuracy                           0.78       665
   macro avg       0.77      0.78      0.77       665
weighted avg       0.79      0.78      0.79       665

Training is ended!
[OK] модель сохранена: models/AlexNet_finetune.pt


Доообучение прошло хорошо. Train accuracy выросло с 65 до 90%, на валидации с 70 до 78%.

Качество нессиметрично по классам - Седан распозднается стабильно лучше внедородника

Попробуем второй вариант (который был на семинаре) дообучать только верхние слои + классификатор.

Замороженные параметры (`requires_grad=False`) оптимизатор просто не обновляет.




In [19]:
model_alexnet_frozen = alexnet(weights=AlexNet_Weights.DEFAULT)

# замораживаем первые 6 параметров f
layers_to_freeze = 6
for i, (name, param) in enumerate(model_alexnet_frozen.features.named_parameters()):
    if i < layers_to_freeze:
        param.requires_grad = False
    print(f'{name:30}{param.requires_grad}') #выводим имя параметра и его статус (будет использоваться или гет)
    #name:30 выравниваение параметров по ширине в 30 символов (чтобы аккуратно было)

model_alexnet_frozen.classifier = torch.nn.Linear(num_in_features, num_out_features)

0.weight                      False
0.bias                        False
3.weight                      False
3.bias                        False
6.weight                      False
6.bias                        False
8.weight                      True
8.bias                        True
10.weight                     True
10.bias                       True


In [20]:
main_kolesa(model_alexnet_frozen)
save_model_artifact(model_alexnet_frozen, 'AlexNet_finetune_frozen')


Epoch: 1


100%|██████████| 97/97 [00:36<00:00,  2.69it/s]



Train set: Average loss: 0.6754, Accuracy: 66.29%
Val set: Average loss: 0.4916, Accuracy: 73.68%

Classification report:
              precision    recall  f1-score   support

       sedan       0.80      0.78      0.79       418
         SUV       0.64      0.67      0.65       247

    accuracy                           0.74       665
   macro avg       0.72      0.72      0.72       665
weighted avg       0.74      0.74      0.74       665


Epoch: 2


100%|██████████| 97/97 [00:35<00:00,  2.74it/s]



Train set: Average loss: 0.5073, Accuracy: 74.70%
Val set: Average loss: 0.4568, Accuracy: 73.38%

Classification report:
              precision    recall  f1-score   support

       sedan       0.83      0.73      0.77       418
         SUV       0.62      0.74      0.68       247

    accuracy                           0.73       665
   macro avg       0.72      0.74      0.72       665
weighted avg       0.75      0.73      0.74       665


Epoch: 3


...


Train set: Average loss: 0.3437, Accuracy: 89.20%
Val set: Average loss: 0.4051, Accuracy: 79.55%

Classification report:
              precision    recall  f1-score   support

       sedan       0.85      0.83      0.84       418
         SUV       0.72      0.74      0.73       247

    accuracy                           0.80       665
   macro avg       0.78      0.79      0.78       665
weighted avg       0.80      0.80      0.80       665


Epoch: 9


100%|██████████| 97/97 [00:36<00:00,  2.66it/s]



Train set: Average loss: 0.2537, Accuracy: 90.72%
Val set: Average loss: 0.4413, Accuracy: 78.65%

Classification report:
              precision    recall  f1-score   support

       sedan       0.86      0.79      0.82       418
         SUV       0.69      0.78      0.73       247

    accuracy                           0.79       665
   macro avg       0.77      0.79      0.78       665
weighted avg       0.80      0.79      0.79       665


Epoch: 10


100%|██████████| 97/97 [00:35<00:00,  2.73it/s]



Train set: Average loss: 0.1109, Accuracy: 92.65%
Val set: Average loss: 0.4188, Accuracy: 80.60%

Classification report:
              precision    recall  f1-score   support

       sedan       0.86      0.82      0.84       418
         SUV       0.72      0.78      0.75       247

    accuracy                           0.81       665
   macro avg       0.79      0.80      0.80       665
weighted avg       0.81      0.81      0.81       665

Training is ended!
[OK] модель сохранена: models/AlexNet_finetune_frozen.pt


Заморозка дала результат по всем метрикам: accuracy +2 п.п., macro-F1 +0.03, и что важно - подтянулся слабый класс «внедорожник» (F1 0.72 → 0.75).

## ResNet18

Возьмём более современную **ResNet18**.

Ее преимущество - остаточные связи, которые позволяют обучать глубокие сети без затухания градиента.

Дообучаем так же - через наш `main_kolesa`, заменив только финальный полносвязный слой.

In [21]:
from torchvision.models import resnet18
from torchvision.models import ResNet18_Weights

model_resnet = resnet18(weights=ResNet18_Weights.DEFAULT)
model_resnet.fc = torch.nn.Linear(model_resnet.fc.in_features, num_out_features)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 139MB/s]


In [22]:
summary(model=model_resnet,
        input_size=(32, 3, 96, 160),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20
)

Layer (type:depth-idx)                   Input Shape          Output Shape         Param #              Trainable
ResNet                                   [32, 3, 96, 160]     [32, 2]              --                   True
├─Conv2d: 1-1                            [32, 3, 96, 160]     [32, 64, 48, 80]     9,408                True
├─BatchNorm2d: 1-2                       [32, 64, 48, 80]     [32, 64, 48, 80]     128                  True
├─ReLU: 1-3                              [32, 64, 48, 80]     [32, 64, 48, 80]     --                   --
├─MaxPool2d: 1-4                         [32, 64, 48, 80]     [32, 64, 24, 40]     --                   --
├─Sequential: 1-5                        [32, 64, 24, 40]     [32, 64, 24, 40]     --                   True
│    └─BasicBlock: 2-1                   [32, 64, 24, 40]     [32, 64, 24, 40]     --                   True
│    │    └─Conv2d: 3-1                  [32, 64, 24, 40]     [32, 64, 24, 40]     36,864               True
│    │    └─BatchN

In [23]:
main_kolesa(model_resnet)
save_model_artifact(model_resnet, 'ResNet18_finetune')


Epoch: 1


100%|██████████| 97/97 [00:36<00:00,  2.63it/s]



Train set: Average loss: 0.7358, Accuracy: 72.90%
Val set: Average loss: 0.3700, Accuracy: 81.50%

Classification report:
              precision    recall  f1-score   support

       sedan       0.86      0.85      0.85       418
         SUV       0.75      0.76      0.75       247

    accuracy                           0.82       665
   macro avg       0.80      0.80      0.80       665
weighted avg       0.82      0.82      0.82       665


Epoch: 2


100%|██████████| 97/97 [00:37<00:00,  2.58it/s]



Train set: Average loss: 0.4949, Accuracy: 86.05%
Val set: Average loss: 0.3832, Accuracy: 83.31%

Classification report:
              precision    recall  f1-score   support

       sedan       0.89      0.83      0.86       418
         SUV       0.75      0.83      0.79       247

    accuracy                           0.83       665
   macro avg       0.82      0.83      0.83       665
weighted avg       0.84      0.83      0.83       665


Epoch: 3


...


Train set: Average loss: 0.0894, Accuracy: 97.45%
Val set: Average loss: 0.3296, Accuracy: 86.77%

Classification report:
              precision    recall  f1-score   support

       sedan       0.89      0.89      0.89       418
         SUV       0.82      0.82      0.82       247

    accuracy                           0.87       665
   macro avg       0.86      0.86      0.86       665
weighted avg       0.87      0.87      0.87       665


Epoch: 9


100%|██████████| 97/97 [00:37<00:00,  2.60it/s]



Train set: Average loss: 0.0821, Accuracy: 98.00%
Val set: Average loss: 0.2410, Accuracy: 87.37%

Classification report:
              precision    recall  f1-score   support

       sedan       0.88      0.93      0.90       418
         SUV       0.87      0.78      0.82       247

    accuracy                           0.87       665
   macro avg       0.87      0.85      0.86       665
weighted avg       0.87      0.87      0.87       665


Epoch: 10


100%|██████████| 97/97 [00:37<00:00,  2.61it/s]



Train set: Average loss: 0.0264, Accuracy: 98.26%
Val set: Average loss: 0.4158, Accuracy: 87.52%

Classification report:
              precision    recall  f1-score   support

       sedan       0.88      0.93      0.90       418
         SUV       0.87      0.78      0.82       247

    accuracy                           0.88       665
   macro avg       0.87      0.86      0.86       665
weighted avg       0.88      0.88      0.87       665

Training is ended!
[OK] модель сохранена: models/ResNet18_finetune.pt


ResNet18 показала лучший результат среди всех архитектур: на валидации accuracy 87.5% и macro-F1 0.86, с уверенным распознаванием обоих классов (sedan F1 0.90, SUV F1 0.82).

Также можно отметить, что модель обучилась быстро, так как к 3-ей эпохе вышло на плато, превосходя значительно остальные модели.

На последней эпохе есть переобучение (98% accuracy на тесте).

## Сравнение и выводы